# 10 - Does the defense generalize to attacks it never saw?

**Question.** Notebook 07b's defense was trained on examples from all three of the paper's attacks. A fair objection: "a model trained on an attack beats one trained on clean data *on that same attack*." Here the same recipe is trained on **TextFooler only**, then tested on all three. BERT-Attack (a masked language model proposes the swaps) and PWWS (a different search) are then attacks the defense has never seen.

**What counts as "never seen".** No BERT-Attack or PWWS example is used anywhere: not in training BERT, not in training the encoder, not in choosing the encoder's epoch. That includes *indirect* use. TextFooler examples that were generated against models which had themselves trained on PWWS or BERT-Attack examples are left out. The only training material is TextFooler run as a black box on training sentences against:
- vanilla BERT;
- the encoder trained without any attack examples (`enc_den`);
- the first-round encoder, trained on TextFooler examples only (`enc_adv`);
- a new round against this notebook's own BERT.

**Recipe:** the one that won in 07b (`advbert2_enc1`), unchanged: the same starting BERT and noise, 2 + 1 epochs of BERT training, the paper's Table 6 GAN, and the encoder with its epoch chosen on attacked *validation* sentences (TextFooler only here). Nothing is chosen on test. It is attacked once on the 1000 test sentences.

**Cost (L4, rough):** about 3-3.5 hours. Everything is cached on Drive and resumes after a disconnect. Run after 07b.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, json, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00-09
N_TEST = 1000
N_ROUND = 2000                     # fresh training sentences for the TextFooler round against this notebook's BERT
ALL = ('textfooler', 'bertattack', 'pwws')
ENC_EPOCHS = 30
W_DEN = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=0.0, w_cls=0.0)   # as in 07b
W_ADV = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=1.0, w_cls=1.0)

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)
baseline = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)     # our TMD

# Start exactly where 07b started (its BERT training records the starting model)
START = json.load(open(lab._path('finetuned_advbert1/DONE')))['init']
RATE = (0.1, 0.3) if START == 'base' else tuple(json.load(open(lab._path('nb07_choice.json')))['rate'])
print('starting BERT, as in 07b:', START, '| noise rate:', RATE)
noise = T.NoiseAugmenter(lab.train_ds['text'], rate=RATE, seed=1)

def texts_from_logs(names):
    # (clean, attacked, label) of every successful attack in these exact log files
    out, seen = [], set()
    for name in names:
        p = lab._path(name)
        assert os.path.exists(p), f'missing {p}'
        for r in map(json.loads, open(p)):
            if r['status'] == 'S' and (r['text'], r['adv_text']) not in seen:
                seen.add((r['text'], r['adv_text']))
                out.append((r['text'], r['adv_text'], r['label']))
    return out

# TextFooler on TRAINING sentences, against victims that never saw any other attack
TF_LOGS = ['advpairs_vanilla_textfooler_n1000_s1.jsonl',      # vanilla BERT
           'advpairs_enc_den_textfooler_n1000_s2.jsonl',      # encoder trained without attack examples
           'advpairs_enc_adv_textfooler_n1000_s3.jsonl']      # encoder trained on TextFooler examples only
VAL_LOGS = ['advpairs_val_vanilla_textfooler_n500_s11.jsonl']  # attacked VALIDATION sentences, for the epoch choice
tf_tr = texts_from_logs(TF_LOGS)
print(len(tf_tr), 'TextFooler-attacked training sentences')

## Step 1: train BERT on clean, noisy and TextFooler-attacked training sentences

In [ ]:
lab.finetune_robust('tfbert1', init=START, epochs=2, lr=1e-5, noise_fn=noise, adv_texts=tf_tr)

## Step 2: attack this BERT with TextFooler on fresh training sentences, then train once more

In [ ]:
lab.use_model('tfbert1')
T.adversarial_pairs(lab, T.Victim(lab, mode='none'), 'tfbert1', 'textfooler', N_ROUND, 41)
tf_tr2 = texts_from_logs(TF_LOGS + [f'advpairs_tfbert1_textfooler_n{N_ROUND}_s41.jsonl'])
print(len(tf_tr2), 'TextFooler-attacked training sentences in total')
lab.finetune_robust('tfbert2', init='tfbert1', epochs=1, lr=1e-5, noise_fn=noise, adv_texts=tf_tr2)

## Step 3: the projection (Table 6 GAN + learned encoder)

In [ ]:
lab.use_model('tfbert2')
Et_tr, _ = lab.embeddings('train', 'tfbert2')
Et_va, _ = lab.embeddings('val', 'tfbert2')
Et_te, _ = lab.embeddings('test', 'tfbert2')
Et_noise, _ = lab.embeddings('train', 'tfbert2', augment_fn=noise, name='train_noise')
gan_t = lab.get_gan('tfbert2', Et_tr, Et_va)
head_t = lab.model.classifier
adv_t = (T.embed_texts(lab, [r[1] for r in tf_tr2]), T.embed_texts(lab, [r[0] for r in tf_tr2]),
         torch.tensor([r[2] for r in tf_tr2]))
val_rows = texts_from_logs(VAL_LOGS)
val_adv_t = (T.embed_texts(lab, [r[1] for r in val_rows]), torch.tensor([r[2] for r in val_rows]))

def get_enc(name, **kw):
    # Train an encoder on this BERT, or load it from Drive
    p = lab._path(f'projenc_{name}_L{gan_t.latent_dim}C{gan_t.code_dim}.pt')
    e = T.ProjEncoder(gan_t.latent_dim, gan_t.code_dim).to(T.DEV)
    if os.path.exists(p):
        e.load_state_dict(torch.load(p)); print('loaded', name)
        return e.eval()
    print('training', name)
    e = T.train_proj_encoder(gan_t, Et_tr, Y_tr, Et_va, Y_va, E_aug=Et_noise, head=head_t, epochs=ENC_EPOCHS, **kw)
    torch.save(e.state_dict(), p)
    return e

enc_den_t = get_enc('tf_enc_den', **W_DEN)
# Guard (the clean-accuracy rule used everywhere): an encoder must keep clean validation accuracy within 1 point of
# BERT's own; otherwise it is retrained with the next seed. The first run (seed 42) collapsed to ~70% clean accuracy.
with torch.no_grad():
    bert_clean = (head_t(Et_va.to(T.DEV)).argmax(-1).cpu() == Y_va).float().mean().item()
def proj_clean(e):
    with torch.no_grad():
        th = T.project(gan_t, Et_va.to(T.DEV), enc=e, mode='encoder')[0]
        return (head_t(th).argmax(-1).cpu() == Y_va).float().mean().item()
for seed in (42, 1, 2, 3):
    name = 'tf_enc1' if seed == 42 else f'tf_enc1_s{seed}'
    enc1_t = get_enc(name, adv=adv_t, init=enc_den_t, select='best', val_adv=val_adv_t, seed=seed, **W_ADV)
    acc = proj_clean(enc1_t)
    print(f'{name}: projected clean val acc {acc:.4f} (BERT alone {bert_clean:.4f})')
    if acc >= bert_clean - 0.01:
        break
else:
    raise RuntimeError('no seed kept clean accuracy: send the log')
TAG = 'tf_only_enc1' if seed == 42 else f'tf_only_enc1_s{seed}'

## The one test run: seen attack (TextFooler) and unseen attacks (BERT-Attack, PWWS)

In [ ]:
lab.use_model('tfbert2')
v = T.Victim(lab, gan=gan_t, enc=enc1_t, mode='tmd', proj_mode='encoder')
T.evaluate(lab, v, TAG, Et_te, Y_te, attacks=ALL, n=N_TEST, need_log=True)
display(T.compare(lab, TAG, baseline_tag='baseline', n=N_TEST))
display(pd.DataFrame([T.mcnemar(lab, 'baseline', TAG, a, N_TEST) for a in ALL]))

P = dict(zip(T.METRICS, T.PAPER['ag_news']['TMD']))
rows = [{'defense': 'paper: TMD', 'clean': P['clean'], 'TextFooler': P['textfooler'], 'BERT-Attack': P['bertattack'],
         'PWWS': P['pwws']}]
for tag, label in [('baseline', 'ours: TMD reproduced'), ('advbert2_enc1', 'ours, trained on all three attacks (07b)'),
                   (TAG, 'ours, trained on TextFooler only')]:
    p = T.results_path(lab, tag, N_TEST)
    if os.path.exists(p):
        r = json.load(open(p))
        rows.append({'defense': label, 'clean': round(r['clean']['acc'], 2),
                     **{T.METRIC_LABEL[a]: round(r[a]['acc'], 1) for a in ALL if a in r}})
display(pd.DataFrame(rows))
print('The last row never saw a BERT-Attack or PWWS example, in training or in choosing its epoch.')

In [ ]:
# Transfer and per-sentence worst case for this defense, saved with notebook 08's results for notebook 09
RES = lab._path('nb08_results.json')
res = json.load(open(RES)) if os.path.exists(RES) else {}
out = [r for r in res.get('worst_case', []) if r['defense'] != TAG]
for attack in ALL:
    src = [r for r in map(json.loads, open(T.examples_path(lab, 'vanilla', N_TEST, attack))) if r['status'] == 'S']
    y, texts = torch.tensor([r['label'] for r in src]), [r['adv_text'] for r in src]
    pred = torch.cat([v.scores(texts[i:i + 64]).argmax(-1).cpu() for i in range(0, len(texts), 64)])
    ok = {r['i']: bool(p) for r, p in zip(src, pred == y)}
    adapt = {r['i']: r['status'] for r in map(json.loads, open(T.examples_path(lab, TAG, N_TEST, attack)))
             if r['status'] != 'D'}
    n = len(adapt)
    out.append(dict(attack=attack, defense=TAG, adaptive=round(100 * sum(s == 'F' for s in adapt.values()) / n, 1),
                    transfer=round(100 * (pred == y).float().mean().item(), 1), transfer_n=len(src),
                    worst_case=round(100 * sum(s == 'F' and ok.get(i, True) for i, s in adapt.items()) / n, 1)))
res['worst_case'] = out
json.dump(res, open(RES, 'w'), indent=1)
display(pd.DataFrame([r for r in out if r['defense'] == TAG]))